# UR-Funny Benchmark

Multimodal humor detection on [UR-Funny](https://github.com/ROC-HCI/UR-FUNNY) (Hasan et al., EMNLP 2019)
using pre-extracted MultiBench features.

> **Note:** This is a quick demonstration, not a SOTA reproduction.

## Data setup

Download `humor.pkl` (~1.1 GB, MultiBench format) and place it at `examples/data/funny/humor.pkl`.

**Option A — manual download:**
```
https://drive.google.com/file/d/1L5slPmYyhEVtwGyM1kgcFMjeBpXLZGT0/view
```

**Option B — automatic via gdown:**
```bash
pip install gdown
```
The notebook will download automatically if `humor.pkl` is not found.

## Experiments

| Experiment | Modalities |
|---|---|
| LinT (A) | audio only (COVAREP, 81-dim) |
| LinT (V) | video only (OpenFace, 371-dim) |
| LinT (T) | text only (GloVe, 300-dim) |
| LinMulT (A+V) | audio + video |
| LinMulT (A+T) | audio + text |
| LinMulT (V+T) | video + text |
| LinMulT (A+V+T) | all three modalities |

Expected test accuracy: ~55–60% for single modalities, ~60–65% for multimodal combinations.

In [ ]:
import pickle
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from linmult import HeadConfig, LinMulT, LinMulTConfig, LinT, LinTConfig

In [ ]:
# ── Config ────────────────────────────────────────────────────────
PKL_FILE = Path("data/funny/humor.pkl")
GDRIVE_ID = "1L5slPmYyhEVtwGyM1kgcFMjeBpXLZGT0"

MAX_SEQ = 20
EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-4

AUDIO_DIM = 81  # COVAREP
VIDEO_DIM = 371  # OpenFace
TEXT_DIM = 300  # GloVe
N_CLASSES = 2
# ─────────────────────────────────────────────────────────────────

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

MOD_DIM = {"audio": AUDIO_DIM, "video": VIDEO_DIM, "text": TEXT_DIM}

print(f"Device  : {DEVICE}")
print(f"Dataset : UR-Funny  ({PKL_FILE})")
print(f"Epochs  : {EPOCHS}  Batch: {BATCH_SIZE}  LR: {LR}\n")

In [ ]:
def _pad_or_trunc(arr: np.ndarray, max_len: int) -> np.ndarray:
    """Return (max_len, F) float32 array."""
    T, F = arr.shape
    n = min(T, max_len)
    x = np.zeros((max_len, F), dtype=np.float32)
    x[:n] = arr[:n]
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)


def load_data() -> dict:
    """Load humor.pkl and return a flat dict of numpy arrays per split."""
    if not PKL_FILE.exists():
        try:
            import gdown

            print("Downloading humor.pkl via gdown ...")
            PKL_FILE.parent.mkdir(parents=True, exist_ok=True)
            gdown.download(id=GDRIVE_ID, output=str(PKL_FILE), quiet=False)
        except ImportError:
            pass

    if not PKL_FILE.exists():
        print(
            f"\n[ERROR] Data file not found: {PKL_FILE}\n\n"
            "Download humor.pkl from Google Drive:\n"
            "  https://drive.google.com/file/d/1L5slPmYyhEVtwGyM1kgcFMjeBpXLZGT0/view\n\n"
            "Or install gdown for automatic download:\n"
            "  pip install gdown\n",
            file=sys.stderr,
        )
        raise FileNotFoundError(PKL_FILE)

    print(f"Loading {PKL_FILE} ...")
    with open(PKL_FILE, "rb") as f:
        raw = pickle.load(f)

    flat = {}
    for split in ("train", "valid", "test"):
        s = raw[split]
        audio_raw = s["audio"]
        video_raw = s["vision"]
        text_raw = s["text"]
        labels = np.array(s["labels"], dtype=np.int64).squeeze()

        N = len(labels)
        audio_X = np.zeros((N, MAX_SEQ, AUDIO_DIM), dtype=np.float32)
        video_X = np.zeros((N, MAX_SEQ, VIDEO_DIM), dtype=np.float32)
        text_X = np.zeros((N, MAX_SEQ, TEXT_DIM), dtype=np.float32)

        iter_a = (
            audio_raw
            if isinstance(audio_raw, np.ndarray) and audio_raw.ndim == 3
            else list(audio_raw)
        )
        iter_v = (
            video_raw
            if isinstance(video_raw, np.ndarray) and video_raw.ndim == 3
            else list(video_raw)
        )
        iter_t = (
            text_raw if isinstance(text_raw, np.ndarray) and text_raw.ndim == 3 else list(text_raw)
        )

        for i in range(N):
            audio_X[i] = _pad_or_trunc(np.asarray(iter_a[i]), MAX_SEQ)
            video_X[i] = _pad_or_trunc(np.asarray(iter_v[i]), MAX_SEQ)
            text_X[i] = _pad_or_trunc(np.asarray(iter_t[i]), MAX_SEQ)

        flat[f"{split}_audio_X"] = audio_X
        flat[f"{split}_video_X"] = video_X
        flat[f"{split}_text_X"] = text_X
        flat[f"{split}_labels"] = labels

    return flat


def make_loaders(data: dict, modalities: list) -> dict:
    loaders = {}
    for split in ("train", "valid", "test"):
        tensors = [torch.from_numpy(data[f"{split}_{mod}_X"]) for mod in modalities]
        tensors.append(torch.from_numpy(data[f"{split}_labels"]))
        ds = TensorDataset(*tensors)
        loaders[split] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=(split == "train"))
    return loaders


_BASE_CFG = dict(
    d_model=40,
    num_heads=4,
    cmt_num_layers=3,
    attention_type="flash",
    time_dim_reducer="gap",
    heads=[HeadConfig(name="cls", type="simple", output_dim=N_CLASSES)],
)


def build_model(modalities: list) -> nn.Module:
    if len(modalities) == 1:
        cfg = LinTConfig(input_feature_dim=MOD_DIM[modalities[0]], **_BASE_CFG)
        return LinT(cfg).to(DEVICE)
    cfg = LinMulTConfig(input_feature_dim=[MOD_DIM[m] for m in modalities], **_BASE_CFG)
    return LinMulT(cfg).to(DEVICE)


def _forward(model, batch, modalities):
    *Xs, labels = batch
    Xs = [x.to(DEVICE) for x in Xs]
    labels = labels.to(DEVICE)
    out = model(Xs[0] if len(modalities) == 1 else Xs)["cls"]
    return out, labels


@torch.no_grad()
def evaluate(model, loader, modalities) -> float:
    model.eval()
    preds, labs = [], []
    for batch in loader:
        out, y = _forward(model, batch, modalities)
        preds.append(out.argmax(1).cpu().numpy())
        labs.append(y.cpu().numpy())
    return float(np.mean(np.concatenate(preds) == np.concatenate(labs)))


def run_experiment(modalities: list, loaders: dict) -> dict:
    model = build_model(modalities)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    best_val_acc, best_state = 0.0, None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in loaders["train"]:
            optimizer.zero_grad()
            out, labels = _forward(model, batch, modalities)
            criterion(out, labels).backward()
            optimizer.step()

        val_acc = evaluate(model, loaders["valid"], modalities)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  epoch {epoch:>3}  val_Acc={val_acc:.1%}")

    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    return {
        split: evaluate(model, loaders[split], modalities) for split in ("train", "valid", "test")
    }


def print_table(results: dict):
    name_w = max(len(n) for n in results) + 2
    col_w = 7
    hdr = f"{'Experiment':<{name_w}}  {'Train':>{col_w}}  {'Valid':>{col_w}}  {'Test':>{col_w}}"
    print()
    print(hdr)
    print("\u2500" * len(hdr))
    for name, m in results.items():
        print(
            f"{name:<{name_w}}  {m['train']:>{col_w}.1%}  \
            {m['valid']:>{col_w}.1%}  {m['test']:>{col_w}.1%}"
        )
    print()

In [ ]:
EXPERIMENTS = [
    ("LinT  (A)", ["audio"]),
    ("LinT  (V)", ["video"]),
    ("LinT  (T)", ["text"]),
    ("LinMulT (A+V)", ["audio", "video"]),
    ("LinMulT (A+T)", ["audio", "text"]),
    ("LinMulT (V+T)", ["video", "text"]),
    ("LinMulT (A+V+T)", ["audio", "video", "text"]),
]

data = load_data()
for split in ("train", "valid", "test"):
    print(f"  {split:5s}: {len(data[f'{split}_labels'])} samples")
print()

In [ ]:
results = {}
for exp_name, modalities in EXPERIMENTS:
    print(f"\u2500\u2500 {exp_name}  {modalities}")
    loaders = make_loaders(data, modalities)
    metrics = run_experiment(modalities, loaders)
    results[exp_name] = metrics
    print(f"  test  Acc={metrics['test']:.1%}\n")

In [ ]:
print_table(results)